# Лабораторная работа 1. Анализ тональности на основе BERT

### Задание 1. Получение оценок качества с классической моделью BERT
В этом задании требуется получить оценки качества на обучающих данных Kaggle (https://www.kaggle.com/c/sentiment-analysis-in-russian/data) с применением модели `RuBERT` на основе отложенной выборки или кросс-валидации.  

При выполнении задания можно воспользоваться предоставленным ноутбуком (`huggingface_bert_finetuning`).  
Подберите гиперпараметры (по крайней мере, количество эпох).  

Выведите результаты и время построения моделей в удобном табличном виде, а также в виде графиков.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers --upgrade

In [3]:
# Импортируем необходимые библиотеки
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import torch
from transformers import BertForSequenceClassification, BertTokenizer, BertConfig, Trainer, TrainingArguments
from datasets import Dataset
import time
import random
from torch.nn.functional import softmax

# Устанавливаем seed для воспроизводимости
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Загрузка данных Kaggle
train_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/train.json'
test_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/test.json'
sample_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/sample.csv'

# Загружаем тренировочные данные
with open(train_path, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# Загружаем тестовые данные
with open(test_path, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# Преобразуем в DataFrame
train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

##Предобработка данных

In [4]:
# Проверим распределение классов
print("Распределение классов в тренировочных данных:")
print(train_df['sentiment'].value_counts())
print(f"\nВсего классов: {train_df['sentiment'].nunique()}")
print(f"Уникальные значения: {train_df['sentiment'].unique()}")

# Создаем маппинг меток для BERT
unique_classes = sorted(train_df['sentiment'].unique())
label_to_id = {label: idx for idx, label in enumerate(unique_classes)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

num_classes = len(unique_classes)  # ← ВАЖНО: определяем num_classes здесь!

print(f"Количество классов: {num_classes}")
print(f"Маппинг меток: {label_to_id}")

# Применяем маппинг к тренировочным данным
train_df['label'] = train_df['sentiment'].map(label_to_id)

Распределение классов в тренировочных данных:
sentiment
neutral     4034
positive    2795
negative    1434
Name: count, dtype: int64

Всего классов: 3
Уникальные значения: ['negative' 'positive' 'neutral']
Количество классов: 3
Маппинг меток: {'negative': 0, 'neutral': 1, 'positive': 2}


##Инциализация модели и токенизация

In [5]:
# Проверяем доступность GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"\nИспользуемое устройство: {device}")

# Выбираем модель
PRE_TRAINED_MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment'

# Инициализируем токенизатор
tokenizer = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

# Анализируем длину текстов
def analyze_text_lengths(df, tokenizer, column='text'):
    token_lengths = []
    for text in df[column]:
        tokens = tokenizer.encode(text, truncation=False)
        token_lengths.append(len(tokens))

    print(f"Минимальная длина: {min(token_lengths)} токенов")
    print(f"Максимальная длина: {max(token_lengths)} токенов")
    print(f"Средняя длина: {np.mean(token_lengths):.2f} токенов")
    print(f"Медианная длина: {np.median(token_lengths):.2f} токенов")

    return token_lengths

print("\nАнализ длины текстов в тренировочных данных:")
train_lengths = analyze_text_lengths(train_df, tokenizer)

# Определяем MAX_LEN
percentile_95 = np.percentile(train_lengths, 95)
MAX_LEN = min(256, int(percentile_95 * 1.1))
print(f"\n95-й перцентиль длины: {percentile_95:.2f}")
print(f"Выбранный MAX_LEN: {MAX_LEN}")


Используемое устройство: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Анализ длины текстов в тренировочных данных:
Минимальная длина: 8 токенов
Максимальная длина: 67116 токенов
Средняя длина: 765.02 токенов
Медианная длина: 408.00 токенов

95-й перцентиль длины: 2271.80
Выбранный MAX_LEN: 256


##Подготовка данных для BERT

In [6]:
# Разделяем данные на тренировочные и валидационные
train_data, val_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=seed,
    stratify=train_df['label']
)

print(f"\nРазмер тренировочного набора: {len(train_data)}")
print(f"Размер валидационного набора: {len(val_data)}")

# Функция для токенизации
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors=None
    )

# Создаем Dataset объекты
train_dataset = Dataset.from_pandas(train_data[['text', 'label']])
val_dataset = Dataset.from_pandas(val_data[['text', 'label']])

# Применяем токенизацию
print("Токенизация данных...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=32)
val_dataset = val_dataset.map(tokenize_function, batched=True, batch_size=32)

# Устанавливаем формат для PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Функция для вычисления метрик
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )
    acc = accuracy_score(labels, preds)

    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)

    return {
        'accuracy': acc,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1.tolist()
    }


Размер тренировочного набора: 7436
Размер валидационного набора: 827
Токенизация данных...


Map:   0%|          | 0/7436 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

##Обучение модели с подбором гиперпараметров

In [7]:
def train_model(num_epochs, learning_rate, batch_size, model_name_suffix=""):
    """Обучение модели с заданными гиперпараметрами"""

    timestamp = int(time.time())
    output_dir = f"./results_{model_name_suffix}_{timestamp}"

    # Упрощенные параметры обучения (совместимые с большинством версий)
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_steps=100,
        logging_dir=f'./logs_{model_name_suffix}_{timestamp}',
        logging_steps=50,
        evaluation_strategy="epoch",  # Пробуем с кавычками
        save_strategy="epoch",
        load_best_model_at_end=True,
        seed=seed,
        report_to='none'
    )

    # Загружаем модель
    print(f"Загружаем модель {PRE_TRAINED_MODEL_NAME}...")

    try:
        model = BertForSequenceClassification.from_pretrained(
            PRE_TRAINED_MODEL_NAME,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
    except:
        # Альтернативный способ загрузки
        config = BertConfig.from_pretrained(PRE_TRAINED_MODEL_NAME)
        config.num_labels = num_classes
        model = BertForSequenceClassification.from_pretrained(
            PRE_TRAINED_MODEL_NAME,
            config=config,
            ignore_mismatched_sizes=True
        )

    model.to(device)

    # Создаем Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    print(f"\n{'='*60}")
    print(f"Обучение модели с параметрами:")
    print(f"  Количество эпох: {num_epochs}")
    print(f"  Learning rate: {learning_rate}")
    print(f"  Batch size: {batch_size}")
    print(f"{'='*60}\n")

    start_time = time.time()
    train_result = trainer.train()
    training_time = time.time() - start_time

    eval_result = trainer.evaluate()

    print(f"\nРезультаты обучения:")
    print(f"  Время обучения: {training_time:.2f} секунд")
    print(f"  Финальные потери: {train_result.metrics['train_loss']:.4f}")
    print(f"\nРезультаты на валидации:")
    print(f"  Accuracy: {eval_result['eval_accuracy']:.4f}")
    print(f"  Macro F1: {eval_result['eval_macro_f1']:.4f}")

    return trainer, eval_result, training_time

## Экспериментируем с разным количеством эпох

In [11]:
# Экспериментируем с разным количеством эпох
results = []
epochs_to_try = [2, 3, 4, 5]

for num_epochs in epochs_to_try:
    print(f"\n{'#'*60}")
    print(f"ЭКСПЕРИМЕНТ: {num_epochs} эпох")
    print(f"{'#'*60}")

    # Обучаем с текущим количеством эпох
    trainer, eval_result, training_time = train_model(
        num_epochs=num_epochs,
        learning_rate=2e-5,
        batch_size=8,
        model_name_suffix=f"epochs_{num_epochs}"
    )

    # Сохраняем результаты
    results.append({
        'epochs': num_epochs,
        'accuracy': eval_result['eval_accuracy'],
        'macro_f1': eval_result['eval_macro_f1'],
        'macro_precision': eval_result['eval_macro_precision'],
        'macro_recall': eval_result['eval_macro_recall'],
        'training_time': training_time
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("Сводная таблица результатов:")
print("="*60)
print(results_df.to_string(index=False))


############################################################
ЭКСПЕРИМЕНТ: 2 эпох
############################################################


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

### Задание 2. Использование BERT как модель для формирования векторных представлений текстов
Основная функция BERT-подобных моделей – формирование контекстно-ориентированных векторных представлений текстов.  
В данном задании нужно будет сделать следующее:
1) обучить RuBERT (как в предыдущем задании);
1) получить из него векторные представления обучающих текстов (например, вектор токена `[CLS]` или среднее векторов всех токенов на последнем слое (см. варианты [здесь](https://mccormickml.com/2019/05/14/BERT-word-embeddings-tutorial/#35-pooling-strategy--layer-choice));
1) применить традиционные модели машинного обучения (логистическая регрессия, SVM, градиентный бустинг и т.п.). Не забывайте про подбор гиперапараметров;
1) получить оценки качества – на таком же разбиении, как в предыдущем задании.

Выведите результаты и время построения моделей в удобном табличном виде, а также в виде графиков.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# Загружаем обученную модель из первого задания
# (предполагаем, что final_model уже обучена)
print("="*60)
print("ЗАДАНИЕ 2: Использование BERT для формирования векторных представлений")
print("="*60)

# 1. Функции для получения векторных представлений из BERT
def get_bert_embeddings(model, tokenizer, texts, max_len=MAX_LEN,
                       method='cls', batch_size=16, device=device):
    """
    Получение эмбеддингов из модели BERT

    Параметры:
    - model: обученная модель BERT
    - tokenizer: токенизатор
    - texts: список текстов
    - max_len: максимальная длина
    - method: метод получения эмбеддингов:
        'cls' - вектор токена [CLS]
        'mean' - среднее всех токенов (исключая паддинг)
        'max' - max pooling всех токенов
    - batch_size: размер батча
    - device: устройство (cpu/gpu)
    """

    model.eval()  # Переводим модель в режим оценки

    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Извлечение эмбеддингов ({method})"):
        batch_texts = texts[i:i+batch_size]

        # Токенизация
        encoded = tokenizer(
            batch_texts,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Перемещаем на устройство
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        # Получаем выходы модели (без вычисления градиентов)
        with torch.no_grad():
            outputs = model.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )

            # Последний скрытый слой [batch_size, seq_len, hidden_size]
            last_hidden_state = outputs.last_hidden_state

            if method == 'cls':
                # Берем эмбеддинг токена [CLS] (первый токен)
                batch_embeddings = last_hidden_state[:, 0, :]

            elif method == 'mean':
                # Среднее по всем токенам (исключая паддинг)
                # Умножаем на attention_mask, чтобы исключить паддинг
                input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
                sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
                sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                batch_embeddings = sum_embeddings / sum_mask

            elif method == 'max':
                # Max pooling по всем токенам
                input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
                # Заменяем эмбеддинги паддинг-токенов на -inf
                last_hidden_state[input_mask_expanded == 0] = -1e9
                batch_embeddings = torch.max(last_hidden_state, 1)[0]

            else:
                raise ValueError(f"Неизвестный метод: {method}")

            # Перемещаем на CPU и преобразуем в numpy
            embeddings.append(batch_embeddings.cpu().numpy())

    # Объединяем все батчи
    return np.vstack(embeddings)

# 2. Получаем эмбеддинги для всех данных
print("\n1. Получение векторных представлений из обученной модели BERT...")

# Получаем тексты и метки
train_texts = train_df['text'].tolist()
train_labels = train_df['label'].values

test_texts = test_df['text'].tolist()

# Получаем эмбеддинги разными методами
print("\nИзвлечение эмбеддингов методом [CLS]...")
X_train_cls = get_bert_embeddings(final_model, tokenizer, train_texts, method='cls', batch_size=16)
X_test_cls = get_bert_embeddings(final_model, tokenizer, test_texts, method='cls', batch_size=16)

print("\nИзвлечение эмбеддингов методом MEAN pooling...")
X_train_mean = get_bert_embeddings(final_model, tokenizer, train_texts, method='mean', batch_size=16)
X_test_mean = get_bert_embeddings(final_model, tokenizer, test_texts, method='mean', batch_size=16)

print("\nИзвлечение эмбеддингов методом MAX pooling...")
X_train_max = get_bert_embeddings(final_model, tokenizer, train_texts, method='max', batch_size=16)
X_test_max = get_bert_embeddings(final_model, tokenizer, test_texts, method='max', batch_size=16)

print(f"\nРазмеры эмбеддингов:")
print(f"  [CLS] эмбеддинги: {X_train_cls.shape}")
print(f"  MEAN эмбеддинги: {X_train_mean.shape}")
print(f"  MAX эмбеддинги: {X_train_max.shape}")

# 3. Разделяем данные на train/validation (так же как в задании 1)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_cls, train_labels, test_size=0.1, random_state=seed, stratify=train_labels
)

print(f"\nРазмеры данных после разделения:")
print(f"  Train: {X_train.shape}, {y_train.shape}")
print(f"  Validation: {X_val.shape}, {y_val.shape}")

# 4. Функция для обучения и оценки традиционных моделей
def evaluate_traditional_models(X_train, X_val, y_train, y_val, embedding_type="CLS"):
    """
    Оценка различных традиционных моделей на эмбеддингах BERT
    """

    results = []
    models = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000,
            random_state=seed,
            n_jobs=-1
        ),
        'SVM': SVC(
            random_state=seed,
            probability=True
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=100,
            random_state=seed,
            n_jobs=-1
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=100,
            random_state=seed
        )
    }

    # Параметры для GridSearch
    param_grids = {
        'LogisticRegression': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l2']
        },
        'SVM': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf']
        },
        'RandomForest': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 10, 20]
        },
        'GradientBoosting': {
            'n_estimators': [50, 100],
            'learning_rate': [0.01, 0.1],
            'max_depth': [3, 5]
        }
    }

    print(f"\n{'='*60}")
    print(f"ОЦЕНКА МОДЕЛЕЙ НА {embedding_type} ЭМБЕДДИНГАХ")
    print(f"{'='*60}")

    for model_name, model in models.items():
        print(f"\nОбучение {model_name}...")

        start_time = time.time()

        if model_name in param_grids:
            # Используем GridSearch для подбора гиперпараметров
            grid_search = GridSearchCV(
                model,
                param_grids[model_name],
                cv=3,
                scoring='f1_macro',
                n_jobs=-1,
                verbose=0
            )

            grid_search.fit(X_train, y_train)

            # Лучшая модель
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_

            # Предсказания
            y_pred = best_model.predict(X_val)
            y_pred_proba = best_model.predict_proba(X_val)

            training_time = time.time() - start_time

            # Метрики
            accuracy = accuracy_score(y_val, y_pred)
            f1_macro = f1_score(y_val, y_pred, average='macro')
            f1_per_class = f1_score(y_val, y_pred, average=None)

            print(f"  Лучшие параметры: {best_params}")
            print(f"  Accuracy: {accuracy:.4f}")
            print(f"  Macro F1: {f1_macro:.4f}")
            print(f"  Время обучения: {training_time:.2f} сек")

            results.append({
                'model': model_name,
                'embedding_type': embedding_type,
                'accuracy': accuracy,
                'macro_f1': f1_macro,
                'f1_per_class': f1_per_class.tolist(),
                'best_params': str(best_params),
                'training_time': training_time
            })
        else:
            # Простое обучение без GridSearch
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            training_time = time.time() - start_time

            accuracy = accuracy_score(y_val, y_pred)
            f1_macro = f1_score(y_val, y_pred, average='macro')
            f1_per_class = f1_score(y_val, y_pred, average=None)

            print(f"  Accuracy: {accuracy:.4f}")
            print(f"  Macro F1: {f1_macro:.4f}")
            print(f"  Время обучения: {training_time:.2f} сек")

            results.append({
                'model': model_name,
                'embedding_type': embedding_type,
                'accuracy': accuracy,
                'macro_f1': f1_macro,
                'f1_per_class': f1_per_class.tolist(),
                'best_params': 'default',
                'training_time': training_time
            })

    return pd.DataFrame(results)

# 5. Оценка моделей на разных типах эмбеддингов
print("\n" + "="*60)
print("СРАВНЕНИЕ РАЗНЫХ ТИПОВ ЭМБЕДДИНГОВ")
print("="*60)

# Подготовка данных для разных типов эмбеддингов
embedding_datasets = {
    'CLS': (X_train_cls, train_labels),
    'MEAN': (X_train_mean, train_labels),
    'MAX': (X_train_max, train_labels)
}

all_results = []

for emb_type, (X_emb, y_all) in embedding_datasets.items():
    print(f"\n\nОбработка {emb_type} эмбеддингов...")

    # Разделяем на train/val
    X_train_emb, X_val_emb, y_train_emb, y_val_emb = train_test_split(
        X_emb, y_all, test_size=0.1, random_state=seed, stratify=y_all
    )

    # Оцениваем модели
    results_df = evaluate_traditional_models(
        X_train_emb, X_val_emb, y_train_emb, y_val_emb, embedding_type=emb_type
    )

    all_results.append(results_df)

# Объединяем все результаты
combined_results = pd.concat(all_results, ignore_index=True)

# 6. Визуализация результатов
print("\n" + "="*60)
print("ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*60)

# Создаем графики
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Сравнение Accuracy по моделям и типам эмбеддингов
accuracy_pivot = combined_results.pivot_table(
    index='model',
    columns='embedding_type',
    values='accuracy'
)
accuracy_pivot.plot(kind='bar', ax=axes[0, 0], width=0.8)
axes[0, 0].set_title('Accuracy по моделям и типам эмбеддингов')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(title='Тип эмбеддинга')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Сравнение Macro F1 по моделям и типам эмбеддингов
f1_pivot = combined_results.pivot_table(
    index='model',
    columns='embedding_type',
    values='macro_f1'
)
f1_pivot.plot(kind='bar', ax=axes[0, 1], width=0.8, color=['red', 'green', 'blue'])
axes[0, 1].set_title('Macro F1 по моделям и типам эмбеддингов')
axes[0, 1].set_ylabel('Macro F1')
axes[0, 1].legend(title='Тип эмбеддинга')
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Время обучения по моделям
time_pivot = combined_results.pivot_table(
    index='model',
    columns='embedding_type',
    values='training_time'
)
time_pivot.plot(kind='bar', ax=axes[1, 0], width=0.8, color=['orange', 'purple', 'brown'])
axes[1, 0].set_title('Время обучения по моделям и типам эмбеддингов')
axes[1, 0].set_ylabel('Время (сек)')
axes[1, 0].set_xlabel('Модель')
axes[1, 0].legend(title='Тип эмбеддинга')
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Heatmap сравнения моделей
heatmap_data = combined_results.pivot_table(
    index='model',
    columns='embedding_type',
    values='macro_f1'
)
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1, 1])
axes[1, 1].set_title('Heatmap: Macro F1 по моделям и типам эмбеддингов')

plt.tight_layout()
plt.show()

# 7. Вывод табличных результатов
print("\n" + "="*60)
print("ТАБЛИЧНЫЕ РЕЗУЛЬТАТЫ")
print("="*60)

# Форматируем вывод
display_df = combined_results[['model', 'embedding_type', 'accuracy', 'macro_f1', 'training_time', 'best_params']].copy()
display_df['accuracy'] = display_df['accuracy'].apply(lambda x: f'{x:.4f}')
display_df['macro_f1'] = display_df['macro_f1'].apply(lambda x: f'{x:.4f}')
display_df['training_time'] = display_df['training_time'].apply(lambda x: f'{x:.2f} сек')

print("\nРезультаты всех экспериментов:")
print(display_df.to_string(index=False))

# 8. Выбор лучшей комбинации
best_result_idx = combined_results['macro_f1'].idxmax()
best_result = combined_results.loc[best_result_idx]

print(f"\n\n{'='*60}")
print("ЛУЧШАЯ КОМБИНАЦИЯ:")
print(f"{'='*60}")
print(f"Модель: {best_result['model']}")
print(f"Тип эмбеддинга: {best_result['embedding_type']}")
print(f"Macro F1: {best_result['macro_f1']:.4f}")
print(f"Accuracy: {best_result['accuracy']:.4f}")
print(f"Время обучения: {best_result['training_time']:.2f} сек")
print(f"Лучшие параметры: {best_result['best_params']}")

# 9. Обучение лучшей модели на всех данных и предсказание на тесте
print(f"\n{'='*60}")
print("ОБУЧЕНИЕ ЛУЧШЕЙ МОДЕЛИ НА ВСЕХ ДАННЫХ И ПРЕДСКАЗАНИЕ НА ТЕСТЕ")
print(f"{'='*60}")

# Выбираем лучшие эмбеддинги
if best_result['embedding_type'] == 'CLS':
    X_all = X_train_cls
    X_test_final = X_test_cls
elif best_result['embedding_type'] == 'MEAN':
    X_all = X_train_mean
    X_test_final = X_test_mean
else:  # MAX
    X_all = X_train_max
    X_test_final = X_test_max

y_all = train_labels

# Определяем лучшую модель
if best_result['model'] == 'LogisticRegression':
    best_model_final = LogisticRegression(
        **eval(best_result['best_params'].replace(':', '=').replace('{', '').replace('}', '')),
        max_iter=1000,
        random_state=seed,
        n_jobs=-1
    )
elif best_result['model'] == 'SVM':
    params = eval(best_result['best_params'].replace(':', '=').replace('{', '').replace('}', ''))
    best_model_final = SVC(
        **params,
        probability=True,
        random_state=seed
    )
elif best_result['model'] == 'RandomForest':
    params = eval(best_result['best_params'].replace(':', '=').replace('{', '').replace('}', ''))
    best_model_final = RandomForestClassifier(
        **params,
        random_state=seed,
        n_jobs=-1
    )
else:  # GradientBoosting
    params = eval(best_result['best_params'].replace(':', '=').replace('{', '').replace('}', ''))
    best_model_final = GradientBoostingClassifier(
        **params,
        random_state=seed
    )

# Обучаем на всех данных
print(f"\nОбучение {best_result['model']} на всех тренировочных данных...")
start_time = time.time()
best_model_final.fit(X_all, y_all)
training_time_final = time.time() - start_time

print(f"Время обучения: {training_time_final:.2f} сек")

# Предсказание на тестовых данных
print("\nПредсказание на тестовых данных...")
test_predictions = best_model_final.predict(X_test_final)
test_predictions_proba = best_model_final.predict_proba(X_test_final)

# Преобразуем обратно в оригинальные метки
test_predicted_sentiments = [id_to_label[class_id] for class_id in test_predictions]

# Создаем submission файл
submission_ml_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/submission_ml.csv'
submission_ml_df = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': test_predicted_sentiments
})
submission_ml_df.to_csv(submission_ml_path, index=False)

print(f"\nSubmission файл сохранен: {submission_ml_path}")
print(f"\nПервые 10 предсказаний:")
print(submission_ml_df.head(10))

# 10. Сравнение с оригинальной BERT моделью
print(f"\n{'='*60}")
print("СРАВНЕНИЕ С ОРИГИНАЛЬНОЙ BERT МОДЕЛЬЮ")
print(f"{'='*60}")

# Для сравнения нужны метрики на валидации для BERT модели
# (берем из результатов первого задания)
print("\nРезультаты BERT модели (из задания 1):")
print(f"  Лучший Macro F1: {results_df.loc[best_result_idx, 'macro_f1']:.4f}")
print(f"  Лучший Accuracy: {results_df.loc[best_result_idx, 'accuracy']:.4f}")

print(f"\nРезультаты {best_result['model']} на {best_result['embedding_type']} эмбеддингах:")
print(f"  Macro F1: {best_result['macro_f1']:.4f}")
print(f"  Accuracy: {best_result['accuracy']:.4f}")

# Визуализация сравнения
comparison_data = pd.DataFrame({
    'Модель': ['BERT (end-to-end)', f"{best_result['model']} + BERT embeddings"],
    'Macro F1': [
        results_df.loc[best_result_idx, 'macro_f1'],
        best_result['macro_f1']
    ],
    'Accuracy': [
        results_df.loc[best_result_idx, 'accuracy'],
        best_result['accuracy']
    ],
    'Время обучения (сек)': [
        results_df.loc[best_result_idx, 'training_time'],
        best_result['training_time']
    ]
})

print(f"\nТаблица сравнения:")
print(comparison_data.to_string(index=False))

# График сравнения
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Macro F1 сравнение
axes[0].bar(comparison_data['Модель'], comparison_data['Macro F1'], color=['blue', 'orange'])
axes[0].set_title('Сравнение Macro F1')
axes[0].set_ylabel('Macro F1')
axes[0].grid(axis='y', alpha=0.3)

# Accuracy сравнение
axes[1].bar(comparison_data['Модель'], comparison_data['Accuracy'], color=['green', 'red'])
axes[1].set_title('Сравнение Accuracy')
axes[1].set_ylabel('Accuracy')
axes[1].grid(axis='y', alpha=0.3)

# Время обучения сравнение
axes[2].bar(comparison_data['Модель'], comparison_data['Время обучения (сек)'], color=['purple', 'brown'])
axes[2].set_title('Сравнение времени обучения')
axes[2].set_ylabel('Время (сек)')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# 11. Сохранение итогового отчета
final_report_task2 = f"""
{'='*80}
ОТЧЕТ ПО ЗАДАНИЮ 2: Использование BERT для формирования векторных представлений
{'='*80}

ИСПОЛЬЗОВАННАЯ BERT МОДЕЛЬ: {PRE_TRAINED_MODEL_NAME}
ТИПЫ ЭМБЕДДИНГОВ: [CLS], MEAN pooling, MAX pooling
ТРАДИЦИОННЫЕ МОДЕЛИ: LogisticRegression, SVM, RandomForest, GradientBoosting

РЕЗУЛЬТАТЫ ЭКСПЕРИМЕНТОВ:
{combined_results[['model', 'embedding_type', 'accuracy', 'macro_f1', 'training_time']].to_string(index=False)}

ЛУЧШАЯ КОМБИНАЦИЯ:
- Модель: {best_result['model']}
- Тип эмбеддингов: {best_result['embedding_type']}
- Macro F1: {best_result['macro_f1']:.4f}
- Accuracy: {best_result['accuracy']:.4f}
- Время обучения: {best_result['training_time']:.2f} сек
- Лучшие параметры: {best_result['best_params']}

СРАВНЕНИЕ С END-TO-END BERT:
- BERT (end-to-end):
  Macro F1: {results_df.loc[best_result_idx, 'macro_f1']:.4f},
  Accuracy: {results_df.loc[best_result_idx, 'accuracy']:.4f}
- {best_result['model']} + BERT embeddings:
  Macro F1: {best_result['macro_f1']:.4f},
  Accuracy: {best_result['accuracy']:.4f}

ВЫВОДЫ:
1. Лучший тип эмбеддингов: {best_result['embedding_type']}
2. Лучшая традиционная модель: {best_result['model']}
3. По сравнению с end-to-end BERT, использование BERT как эмбеддингов + традиционные модели:
   - {"улучшило" if best_result['macro_f1'] > results_df.loc[best_result_idx, 'macro_f1'] else "ухудшило"} качество на {abs(best_result['macro_f1'] - results_df.loc[best_result_idx, 'macro_f1']):.4f}
   - {"быстрее" if best_result['training_time'] < results_df.loc[best_result_idx, 'training_time'] else "медленнее"} на {abs(best_result['training_time'] - results_df.loc[best_result_idx, 'training_time']):.2f} сек

SUBMISSION ФАЙЛЫ:
1. BERT end-to-end: {submission_path}
2. {best_result['model']} + BERT embeddings: {submission_ml_path}

ДАТА ВЫПОЛНЕНИЯ: {time.strftime('%Y-%m-%d %H:%M:%S')}
{'='*80}
"""

print("\n" + final_report_task2)

# Сохраняем отчет
report_task2_path = '/content/drive/MyDrive/DL/DL_labs/dl_lab_1/report_task2.txt'
with open(report_task2_path, 'w', encoding='utf-8') as f:
    f.write(final_report_task2)

print(f"Отчет по заданию 2 сохранен: {report_task2_path}")

ЗАДАНИЕ 2: Использование BERT для формирования векторных представлений

1. Получение векторных представлений из обученной модели BERT...

Извлечение эмбеддингов методом [CLS]...


NameError: name 'final_model' is not defined

### Задание 3. Оценка качества классификации на Kaggle
Выберите лучшую модель, обучите её на всем обучающем корпусе, получите предсказания для тестовых данных и отправьте результаты на Kaggle.  

Сравните полученные результаты с результатами на Kaggle: https://www.kaggle.com/c/sentiment-analysis-in-russian/leaderboard.

Выведите кроме `macro F1-score` следующие метрики для отложенной выборки:
- `F1-score` по каждому классу;
- `Precision` и `Recall` по каждому классу;
- `Confusion matrix` по каждому классу и для всей выборки.

Сделайте выводы в целом по своим исследованиям – приведите в одной таблице несколько лучших моделей с указанием параметров и времени работы.

In [ ]:
# Ваш код здесь